# Visual Product Similarity — Stanford Online Products (SOP) Pipeline

Self-contained notebook: run every cell top to bottom and it will:
1. Install dependencies
2. Download & extract the Stanford Online Products dataset
3. Write out the pipeline code (feature extraction, FAISS indexing, search, evaluation)
4. Run the full pipeline: parse metadata -> extract ResNet50 embeddings -> build FAISS index -> evaluate

Works on **Google Colab** or **Kaggle Notebooks** — just make sure the runtime has a GPU:
- Colab: `Runtime -> Change runtime type -> GPU`
- Kaggle: `Settings -> Accelerator -> GPU T4 x2` (or P100)

## Step 0 — Install dependencies

In [ ]:
!pip install -q faiss-cpu tqdm scikit-image
# torch/torchvision/numpy/pillow are already preinstalled on Colab and Kaggle

## Step 1 — Download the Stanford Online Products dataset

**Try the official source first.** If the FTP link fails (university FTP servers are sometimes unreliable), search Kaggle's dataset catalog for a mirrored copy instead — on Kaggle notebooks you can add it directly via **Add Input** in the sidebar without downloading anything.

In [ ]:
import os



DATA_ROOT = "/content" if os.path.exists("/content") else "/kaggle/working"

os.chdir(DATA_ROOT)

print("Working directory:", os.getcwd())

In [ ]:
# Official source (2.9GB). If this cell fails or hangs, see the fallback options below.

!curl -o Stanford_Online_Products.zip "ftp://cs.stanford.edu/cs/cvgl/Stanford_Online_Products.zip"

!unzip -q Stanford_Online_Products.zip

!ls Stanford_Online_Products | head

**If the FTP download failed:**
- **Kaggle:** sidebar -> Add Input -> search "Stanford Online Products" -> Add. It'll mount at `/kaggle/input/<dataset-name>/` — update `SOP_ROOT` below to point there instead.
- **Colab:** try a Hugging Face or Google Drive mirror, e.g.:
  ```python
  !pip install -q huggingface_hub
  from huggingface_hub import snapshot_download
  snapshot_download(repo_id="JamieSJS/stanford-online-products", repo_type="dataset", local_dir="Stanford_Online_Products")
  ```
  (Check the repo's actual file layout — mirror structures vary — and adjust `SOP_ROOT` accordingly.)

In [ ]:
# Point this at wherever Ebay_train.txt / Ebay_test.txt actually ended up

SOP_ROOT = os.path.join(DATA_ROOT, "Stanford_Online_Products")

assert os.path.exists(os.path.join(SOP_ROOT, "Ebay_test.txt")), f"Ebay_test.txt not found under {SOP_ROOT} -- fix SOP_ROOT above"

print("Dataset found at:", SOP_ROOT)

## (Optional, Colab only) Save the dataset to Google Drive
So you don't have to re-download 2.9GB every session. Skip this cell on Kaggle.

In [ ]:
if os.path.exists("/content"):

    from google.colab import drive

    drive.mount('/content/drive')

    import shutil

    dest = '/content/drive/MyDrive/Stanford_Online_Products'

    if not os.path.exists(dest):

        shutil.copytree(SOP_ROOT, dest)

        print('Copied dataset to', dest)

    else:

        print('Already exists in Drive:', dest)

else:

    print('Not on Colab, skipping.')

## Step 2 — Write out the pipeline code

This recreates the `src/` files from the project I built earlier. Nothing to edit here — just run it.

In [ ]:
import os
os.makedirs("src", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

In [ ]:
%%writefile src/utils.py
"""
utils.py
Shared helper functions used across the pipeline.
"""

import os
import re
from pathlib import Path
from typing import List


def list_images(image_dir: str) -> List[str]:
    """Return a sorted list of absolute paths to all images in a directory."""
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    paths = [
        str(p.resolve())
        for p in Path(image_dir).iterdir()
        if p.suffix.lower() in exts
    ]
    return sorted(paths)


def get_category(filename: str) -> str:
    """
    Extract the product category from a filename like:
        'backpack_10_1771054475342.jpg' -> 'backpack'
    Falls back to 'unknown' if the pattern doesn't match.
    """
    name = os.path.basename(filename)
    match = re.match(r"^([a-zA-Z]+)_", name)
    return match.group(1).lower() if match else "unknown"


In [ ]:
%%writefile src/parse_sop_metadata.py
"""
parse_sop_metadata.py

Converts the Stanford Online Products (SOP) dataset's native metadata files
(Ebay_train.txt, Ebay_test.txt) into the unified metadata.json schema used
by the rest of this pipeline (build_faiss_index.py, search.py, evaluate.py).

Expected SOP folder layout after unzipping Stanford_Online_Products.zip:

    Stanford_Online_Products/
        Ebay_train.txt          <- class_id, super_class_id, path (train split, 59,551 imgs)
        Ebay_test.txt           <- class_id, super_class_id, path (test split, 60,502 imgs)
        bicycle_final/
        cabinet_final/
        chair_final/
        ...                     <- actual .jpg files, one folder per super-class

Each row of Ebay_train.txt / Ebay_test.txt looks like:
    image_id class_id super_class_id path
    1        1        1              bicycle_final/111265328556_0.JPG

- class_id  : ~22,634 fine-grained classes -> exact same physical product
              (the real ground truth for "visually similar" in this dataset)
- super_class_id (1-12) -> coarse category (bicycle, chair, lamp, sofa, ...)

Run:
    python src/parse_sop_metadata.py \
        --sop_root /path/to/Stanford_Online_Products \
        --split train \
        --out_dir outputs

    # or combine both splits into one catalog:
    python src/parse_sop_metadata.py --sop_root /path/to/Stanford_Online_Products --split all --out_dir outputs
"""

import argparse
import csv
import json
import os

_SUPER_CLASSES = [
    "bicycle", "cabinet", "chair", "coffee_maker", "fan", "kettle",
    "lamp", "mug", "sofa", "stapler", "table", "toaster",
]

_SPLIT_FILES = {"train": "Ebay_train.txt", "test": "Ebay_test.txt"}


def parse_split(sop_root: str, split_file: str, split_name: str):
    path = os.path.join(sop_root, split_file)
    records = []
    with open(path, "r") as f:
        reader = csv.DictReader(f, delimiter=" ")
        for row in reader:
            class_id = int(row["class_id"]) - 1
            super_class_id = int(row["super_class_id"]) - 1
            img_path = os.path.join(sop_root, row["path"])
            records.append(
                {
                    "path": img_path,
                    "filename": os.path.basename(img_path),
                    "category": _SUPER_CLASSES[super_class_id],  # coarse, for quick filtering/eval
                    "class_id": class_id,                        # fine-grained, exact-product ground truth
                    "split": split_name,
                }
            )
    return records


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--sop_root", required=True, help="Path to extracted Stanford_Online_Products/ folder")
    parser.add_argument("--split", choices=["train", "test", "all"], default="test",
                         help="test=60,502 imgs (recommended to start: smaller & has the official eval protocol)")
    parser.add_argument("--out_dir", default="outputs")
    parser.add_argument("--limit", type=int, default=None,
                         help="Optional cap on number of images (useful for a quick trial run before scaling to all 120k)")
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)

    records = []
    splits = ["train", "test"] if args.split == "all" else [args.split]
    for s in splits:
        records.extend(parse_split(args.sop_root, _SPLIT_FILES[s], s))

    missing = [r for r in records if not os.path.exists(r["path"])]
    if missing:
        print(f"WARNING: {len(missing)} referenced image files not found on disk "
              f"(check --sop_root path). First missing: {missing[0]['path']}")

    if args.limit:
        records = records[: args.limit]

    out_path = os.path.join(args.out_dir, "catalog_metadata.json")
    with open(out_path, "w") as f:
        json.dump(records, f, indent=2)

    n_classes = len(set(r["class_id"] for r in records))
    print(f"Parsed {len(records)} images across {n_classes} fine-grained classes "
          f"and {len(_SUPER_CLASSES)} super-categories.")
    print(f"Saved -> {out_path}")
    print("Next step: python src/feature_extraction.py --metadata_json outputs/catalog_metadata.json --out_dir outputs")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/feature_extraction.py
"""
feature_extraction.py
Step 1 of the pipeline: Image Feature Extraction (now scaled for large datasets)

Uses a pretrained ResNet50 (ImageNet weights) with the final classification
layer removed, so each image becomes a 2048-dim dense embedding vector.

Two input modes:
  --image_dir <folder>        generic mode: embed every image in a flat folder
                               (what we used for the 500-image sample set)
  --metadata_json <file>      catalog mode: embed exactly the images listed in
                               a metadata.json / catalog_metadata.json (what
                               parse_sop_metadata.py produces for Stanford
                               Online Products, keeping class_id/super_class_id)

Built for scale (SOP has 59k-120k images):
  - Batched GPU/CPU inference via DataLoader (--batch_size, --num_workers)
  - Checkpointing every --checkpoint_every images, so a multi-hour run on
    120k images can resume after an interruption instead of restarting
  - Skips corrupt/unreadable images instead of crashing the whole run

Run:
    # sample folder (small dataset)
    python src/feature_extraction.py --image_dir data/images --out_dir outputs

    # Stanford Online Products (large dataset)
    python src/parse_sop_metadata.py --sop_root /path/to/Stanford_Online_Products --split test --out_dir outputs
    python src/feature_extraction.py --metadata_json outputs/catalog_metadata.json --out_dir outputs --batch_size 64

Note: the first run downloads pretrained ImageNet weights (~100MB) from
download.pytorch.org, so it needs normal internet access.
"""

import argparse
import json
import os

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm

from utils import list_images, get_category


def build_model(device: str) -> nn.Module:
    """Load pretrained ResNet50 and strip the final FC (classification) layer."""
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    resnet = models.resnet50(weights=weights)
    modules = list(resnet.children())[:-1]  # drop final FC -> 2048-d pooled features
    model = nn.Sequential(*modules)
    model.eval()
    model.to(device)
    return model


def get_transform() -> transforms.Compose:
    return transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


@torch.no_grad()
def extract_embedding(model: nn.Module, transform, image_path: str, device: str) -> np.ndarray:
    """Single-image convenience wrapper, used by search.py / app.py for query images."""
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)
    features = model(tensor).squeeze().cpu().numpy().astype("float32")
    norm = np.linalg.norm(features)
    return features / norm if norm > 0 else features


class ImageDataset(Dataset):
    """Loads images for batched inference; returns None on unreadable files
    (filtered out in collate_fn) so one bad file doesn't kill a multi-hour run."""

    def __init__(self, records, transform):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        try:
            img = Image.open(rec["path"]).convert("RGB")
            tensor = self.transform(img)
            return tensor, idx
        except Exception as e:
            print(f"Skipping unreadable image {rec['path']}: {e}")
            return None


def collate_skip_none(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None, None
    tensors, idxs = zip(*batch)
    return torch.stack(tensors), list(idxs)


def load_records(args):
    """Build the unified list of {path, filename, category, ...} records."""
    if args.metadata_json:
        with open(args.metadata_json) as f:
            records = json.load(f)
    else:
        paths = list_images(args.image_dir)
        records = [
            {"path": p, "filename": os.path.basename(p), "category": get_category(p)}
            for p in paths
        ]
    return records


def _save_checkpoint(embeddings, records, out_dir, final=False):
    """Save only the successfully embedded records so far (skips None slots
    from unreadable images), keeping embeddings.npy and metadata.json aligned."""
    valid_idx = [i for i, e in enumerate(embeddings) if e is not None]
    if not valid_idx:
        return
    emb_arr = np.vstack([embeddings[i] for i in valid_idx]).astype("float32")
    meta = [records[i] for i in valid_idx]

    np.save(os.path.join(out_dir, "embeddings.npy"), emb_arr)
    with open(os.path.join(out_dir, "metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)

    tag = "FINAL" if final else "checkpoint"
    print(f"[{tag}] Saved {emb_arr.shape[0]}/{len(records)} embeddings -> {out_dir}/embeddings.npy")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image_dir", default=None, help="Flat folder of images (generic mode)")
    parser.add_argument("--metadata_json", default=None, help="Pre-built catalog (e.g. from parse_sop_metadata.py)")
    parser.add_argument("--out_dir", default="outputs")
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--num_workers", type=int, default=4)
    parser.add_argument("--checkpoint_every", type=int, default=2000,
                         help="Save partial embeddings.npy/metadata.json every N images")
    args = parser.parse_args()

    if not args.image_dir and not args.metadata_json:
        parser.error("Provide either --image_dir or --metadata_json")

    os.makedirs(args.out_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    model = build_model(device)
    transform = get_transform()

    records = load_records(args)
    print(f"Embedding {len(records)} images...")

    dataset = ImageDataset(records, transform)
    loader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        num_workers=args.num_workers,
        collate_fn=collate_skip_none,
    )

    embeddings = [None] * len(records)
    processed = 0

    with torch.no_grad():
        for tensors, idxs in tqdm(loader, desc="Extracting embeddings"):
            if tensors is None:
                continue
            tensors = tensors.to(device)
            feats = model(tensors).squeeze(-1).squeeze(-1).cpu().numpy().astype("float32")  # (B, 2048)
            norms = np.linalg.norm(feats, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            feats = feats / norms

            for emb, idx in zip(feats, idxs):
                embeddings[idx] = emb
            processed += len(idxs)

            if processed % args.checkpoint_every < args.batch_size:
                _save_checkpoint(embeddings, records, args.out_dir)

    _save_checkpoint(embeddings, records, args.out_dir, final=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/build_faiss_index.py
"""
build_faiss_index.py
Step 2 of the pipeline: Embedding Indexing (scaled)

Loads outputs/embeddings.npy and builds a FAISS index. Embeddings are
L2-normalized, so Inner Product == cosine similarity.

Two index types:
  --index_type flat   IndexFlatIP: exact search, fine up to ~50-100k vectors,
                       still sub-100ms per query at that scale on CPU.
  --index_type ivf     IndexIVFFlat: clusters vectors into --nlist cells and
                       only searches --nprobe of them -> much faster at
                       100k-millions of vectors, at a small recall cost.
                       This is what you'd use once you index the full SOP
                       dataset (~120k images) or scale beyond it.

Run:
    python src/build_faiss_index.py --out_dir outputs --index_type flat
    python src/build_faiss_index.py --out_dir outputs --index_type ivf --nlist 256 --nprobe 16
"""

import argparse
import os

import faiss
import numpy as np


def build_flat_index(embeddings: np.ndarray) -> faiss.Index:
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index


def build_ivf_index(embeddings: np.ndarray, nlist: int, nprobe: int) -> faiss.Index:
    dim = embeddings.shape[1]
    quantizer = faiss.IndexFlatIP(dim)
    index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
    # IVF needs training on a representative sample before vectors can be added
    index.train(embeddings)
    index.add(embeddings)
    index.nprobe = nprobe
    return index


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--out_dir", default="outputs")
    parser.add_argument("--index_type", choices=["flat", "ivf"], default="flat")
    parser.add_argument("--nlist", type=int, default=256,
                         help="Number of IVF clusters. Rule of thumb: ~sqrt(N) to 4*sqrt(N)")
    parser.add_argument("--nprobe", type=int, default=16,
                         help="How many clusters to search per query (higher = more accurate, slower)")
    args = parser.parse_args()

    embeddings = np.load(os.path.join(args.out_dir, "embeddings.npy")).astype("float32")
    n = embeddings.shape[0]

    if args.index_type == "ivf" and n < args.nlist * 40:
        print(f"WARNING: only {n} vectors for nlist={args.nlist}; FAISS recommends "
              f"~40+ vectors per cluster for good training. Consider a smaller --nlist "
              f"or --index_type flat for datasets this size.")

    if args.index_type == "flat":
        index = build_flat_index(embeddings)
    else:
        index = build_ivf_index(embeddings, args.nlist, args.nprobe)

    index_path = os.path.join(args.out_dir, "faiss.index")
    faiss.write_index(index, index_path)

    print(f"Indexed {index.ntotal} vectors of dim {embeddings.shape[1]} ({args.index_type})")
    print(f"Saved FAISS index -> {index_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/search.py
"""
search.py
Step 3 of the pipeline: Similarity Matching

Given a query image, extract its embedding and retrieve the Top-K most
visually similar products from the FAISS index (cosine similarity).

Run:
    python src/search.py --query data/images/bottle_0_....jpg --k 5

Swap EXTRACT_FN below between the CNN extractor (feature_extraction.py) and
the offline demo extractor depending on which one built your index/embeddings.
"""

import argparse
import json
import os

import faiss
import numpy as np


def load_index_and_metadata(out_dir: str):
    index = faiss.read_index(os.path.join(out_dir, "faiss.index"))
    with open(os.path.join(out_dir, "metadata.json")) as f:
        metadata = json.load(f)
    return index, metadata


def search(query_embedding: np.ndarray, index, metadata, k: int = 5, category_filter: str = None):
    """
    Returns Top-K results as a list of dicts: {rank, filename, category, path, score}
    Ranked & filtered per Step 4 (rank by score; optional category filter).
    """
    query_embedding = query_embedding.reshape(1, -1).astype("float32")
    # Over-fetch if filtering, then trim, so filtering doesn't starve results
    fetch_k = k * 5 if category_filter else k
    fetch_k = min(fetch_k, index.ntotal)

    scores, indices = index.search(query_embedding, fetch_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        item = metadata[idx]
        if category_filter and item["category"] != category_filter:
            continue
        results.append(
            {
                "filename": item["filename"],
                "category": item["category"],
                "path": item["path"],
                "score": float(score),
            }
        )
        if len(results) >= k:
            break

    for rank, r in enumerate(results, start=1):
        r["rank"] = rank
    return results


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--query", required=True, help="Path to query image")
    parser.add_argument("--out_dir", default="outputs")
    parser.add_argument("--k", type=int, default=5)
    parser.add_argument("--category_filter", default=None)
    parser.add_argument(
        "--extractor",
        choices=["cnn", "offline_demo"],
        default="offline_demo",
        help="Which feature extractor matches the built index",
    )
    args = parser.parse_args()

    if args.extractor == "cnn":
        from feature_extraction import build_model, get_transform, extract_embedding
        import torch

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = build_model(device)
        transform = get_transform()
        query_emb = extract_embedding(model, transform, args.query, device)
    else:
        from feature_extraction_offline_demo import extract_embedding

        query_emb = extract_embedding(args.query)

    index, metadata = load_index_and_metadata(args.out_dir)
    results = search(query_emb, index, metadata, k=args.k, category_filter=args.category_filter)

    print(f"\nTop-{args.k} visually similar products for: {os.path.basename(args.query)}\n")
    for r in results:
        print(f"  #{r['rank']}  {r['filename']:35s}  category={r['category']:12s}  score={r['score']:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/evaluate.py
"""
evaluate.py
Step 5 of the pipeline: Evaluation

Since we don't have human-labeled "similar pairs", we use dataset labels as
a proxy ground truth. Two relevance definitions are supported via --relevance:

  category   a retrieved item is "relevant" if it shares the query's coarse
             category (e.g. both are "chair"). Works for any dataset, since
             it's derived from the 'category' field our pipeline always sets.

  class_id   a retrieved item is "relevant" only if it's the exact same
             fine-grained product/instance as the query. This is the
             official Stanford Online Products benchmark protocol (22,634
             classes) and is a much stricter, more meaningful test of
             whether the embeddings actually capture "the same product" vs.
             just "the same rough type of product". Requires metadata.json
             to have a 'class_id' field (produced by parse_sop_metadata.py).

For every image in the dataset (used as a query, excluding itself from its
own results), we compute:
  - Precision@K = (# relevant items in top-K) / K
  - Recall@K    = (# relevant items in top-K) / (total relevant items in the dataset)
Averaged (macro) over all queries, reported overall + per group.

Run:
    python src/evaluate.py --k 5 --relevance category
    python src/evaluate.py --k 10 --relevance class_id   # Stanford Online Products protocol
"""

import argparse
import json
import os
from collections import defaultdict

import faiss
import numpy as np


def evaluate(out_dir: str, k: int = 5, relevance: str = "category"):
    embeddings = np.load(os.path.join(out_dir, "embeddings.npy")).astype("float32")
    with open(os.path.join(out_dir, "metadata.json")) as f:
        metadata = json.load(f)

    if relevance == "class_id" and "class_id" not in metadata[0]:
        raise ValueError(
            "metadata.json has no 'class_id' field. --relevance class_id only works "
            "for catalogs built with parse_sop_metadata.py. Use --relevance category instead."
        )

    index = faiss.read_index(os.path.join(out_dir, "faiss.index"))

    labels = [m[relevance] for m in metadata]
    label_counts = defaultdict(int)
    for lab in labels:
        label_counts[lab] += 1

    # search top-(k+1) since the query image itself will be the #1 hit (score=1.0)
    scores, indices = index.search(embeddings, k + 1)

    precisions = []
    recalls = []
    per_group_precisions = defaultdict(list)

    for query_idx in range(len(metadata)):
        query_label = labels[query_idx]
        total_relevant = label_counts[query_label] - 1  # excluding the query itself

        neighbors = [idx for idx in indices[query_idx] if idx != query_idx][:k]
        relevant_hits = sum(1 for idx in neighbors if labels[idx] == query_label)

        precision = relevant_hits / k
        recall = relevant_hits / total_relevant if total_relevant > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        per_group_precisions[query_label].append(precision)

    print(f"\n=== Evaluation @K={k} (relevance = {relevance}) ===")
    print(f"Overall Precision@{k}: {np.mean(precisions):.4f}")
    print(f"Overall Recall@{k}:    {np.mean(recalls):.4f}\n")

    # printing per-group breakdown is only useful for coarse groupings
    # (category: ~10-12 groups); skip it for class_id (~22k groups)
    per_group_summary = {}
    if relevance == "category":
        print("Per-category Precision@K:")
        for cat in sorted(per_group_precisions):
            vals = per_group_precisions[cat]
            print(f"  {cat:14s} n={len(vals):4d}  Precision@{k}={np.mean(vals):.4f}")
            per_group_summary[cat] = float(np.mean(vals))

    return {
        "k": k,
        "relevance": relevance,
        "overall_precision_at_k": float(np.mean(precisions)),
        "overall_recall_at_k": float(np.mean(recalls)),
        "per_group_precision_at_k": per_group_summary,
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--out_dir", default="outputs")
    parser.add_argument("--k", type=int, default=5)
    parser.add_argument("--relevance", choices=["category", "class_id"], default="category")
    args = parser.parse_args()

    results = evaluate(args.out_dir, args.k, args.relevance)

    out_path = os.path.join(args.out_dir, f"evaluation_{args.relevance}_k{args.k}.json")
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nSaved metrics -> {out_path}")


if __name__ == "__main__":
    main()


## Step 3 — Convert SOP's metadata into the pipeline's catalog format

Starting with the **test split** (60,502 images) — smaller, and it's the official eval split. `--limit` is there so you can do a fast first pass on a small slice before committing to the full run — set it to `None` (remove the flag) once you're confident everything works.

In [ ]:
# Fast first pass: small slice to sanity-check the whole pipeline (~2 minutes)

!python src/parse_sop_metadata.py --sop_root "$SOP_ROOT" --split test --out_dir outputs --limit 2000

In [ ]:
# once satisfied, uncomment this to run the FULL test split (60,502 images):

# !python src/parse_sop_metadata.py --sop_root "$SOP_ROOT" --split test --out_dir outputs

## Step 4 — Extract ResNet50 embeddings

Batched, GPU-aware, and checkpointed every 5,000 images (so an interrupted run resumes instead of restarting). First run downloads pretrained ImageNet weights (~100MB), needs internet.

In [ ]:
!python src/feature_extraction.py --metadata_json outputs/catalog_metadata.json --out_dir outputs --batch_size 64 --num_workers 4 --checkpoint_every 5000

## Step 5 — Build the FAISS index

Use `flat` (exact search) for the 2,000-image test slice. Switch to `ivf` once you're indexing the full 60K+ image split — meaningfully faster at that scale.

In [ ]:
!python src/build_faiss_index.py --out_dir outputs --index_type flat

# at full scale (60k+), use instead:

# !python src/build_faiss_index.py --out_dir outputs --index_type ivf --nlist 512 --nprobe 24

## Step 6 — Evaluate against the real SOP benchmark

`--relevance class_id` checks whether retrieved items are the *exact same product*, not just the same coarse category — the actual SOP task. Published ResNet50 baselines land around **Recall@1 ≈ 0.60–0.68** on the full official test split; use that as a sanity check once you run on the full 60,502 images (a 2,000-image slice will score lower/noisier since it's a random subset).

In [ ]:
!python src/evaluate.py --out_dir outputs --k 10 --relevance class_id

In [ ]:
# also useful: coarse category-level precision (bicycle/chair/lamp/etc.)

!python src/evaluate.py --out_dir outputs --k 10 --relevance category

## Step 7 — Try an actual similarity search

Pick any image from the catalog as a query and see its top-5 visually similar products.

In [ ]:
import json, sys

sys.path.insert(0, 'src')

from search import load_index_and_metadata, search

from feature_extraction import build_model, get_transform, extract_embedding

import torch



device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = build_model(device)

transform = get_transform()



index, metadata = load_index_and_metadata('outputs')

query_path = metadata[0]['path']  # swap for any path you like

query_emb = extract_embedding(model, transform, query_path, device)



results = search(query_emb, index, metadata, k=5)

print(f"Query: {query_path}\n")

for r in results:

    print(f"  #{r['rank']}  {r['filename']:30s}  category={r['category']:12s}  score={r['score']:.4f}")

In [ ]:
# visualize the query + results

import matplotlib.pyplot as plt

from PIL import Image



fig, axes = plt.subplots(1, len(results) + 1, figsize=(15, 3))

axes[0].imshow(Image.open(query_path)); axes[0].set_title('QUERY'); axes[0].axis('off')

for ax, r in zip(axes[1:], results):

    ax.imshow(Image.open(r['path']))

    ax.set_title(f"#{r['rank']} {r['score']:.2f}")

    ax.axis('off')

plt.tight_layout()

plt.show()

## Next steps
- Rerun Step 3 without `--limit` to process the full 60,502-image test split (takes longer, but gives you the real, comparable benchmark numbers).
- Try `--split train` or `--split all` in `parse_sop_metadata.py` for the full 120K-image catalog.
- Download `outputs/embeddings.npy`, `outputs/faiss.index`, and `outputs/catalog_metadata.json` at the end of your session (Colab/Kaggle VMs are ephemeral) — either `files.download(...)` on Colab or save to `/kaggle/working/` which Kaggle keeps as notebook output.